In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
import json
import sqlite3
from scipy import stats
from geopy.distance import geodesic
from geopy.geocoders import OpenCage
import geocoder
import os

os.chdir("C:\\Users\\serojans\\hydro_SGU_SMHI_return_periods\\src")
from sgu import *
from smhi import *
from coordinate_converter import *
from telecontrolnet import *
from projektnavet import *
from groundwater_level_plotter import *
from groundwater_calculation import *


os.chdir("C:\\Users\\serojans\\hydro_SGU_SMHI_return_periods\\metadata")
sgu = fetch_sgu_data()
smhi = fetch_smhi_data()
telecontrolnet = fetch_telecontrolnet_data()
convert_coordinate = convert_coordinate()
projektnavet = fetch_projektnavet_data()
groundwater_return_periods = groundwater_return_periods()
# groundwater_plotter = groundwater_level_plotter()


In [ ]:
# metadata = projektnavet.extract_metadata()
# level_data_1 = projektnavet.extract_leveldata('2019-01-01', '2022-02-20')
# level_data_2 = projektnavet.extract_leveldata('2022-02-20', '2024-02-22')

# level_data = pd.concat([level_data_1, level_data_2])

# with pd.ExcelWriter('Projektnav_object_data.xlsx') as writer:
#     metadata.to_excel(writer, index=False)
# with pd.ExcelWriter('Projektnav_level_data.xlsx') as writer:
#     level_data.to_excel(writer, index=False)

metadata = pd.read_excel("Projektnav_object_data.xlsx")
level_data = pd.read_excel("Projektnav_level_data.xlsx")

observation_wells = level_data.loc[level_data.Nr.isin(['AA7571U', 'AA7611U', 'AA7573U', 'KA4106U', 'AA7574U', 'AA7570U', 'AA4113U'])]
observation_wells = projektnavet.map_metadata_to_leveldata(observation_wells, metadata)

# observation_wells.head(2)

In [ ]:
# sgu_stations = sgu.fetch_stations(lanskod=14)
# sgu_data = sgu.fetch_measurements(sgu_stations, is_active=True)
# sgu_standard = geolocator.sweref_to_lat_lon(sgu_data)
# sgu_data_modify = sgu.convert_columns_to_standard(sgu_standard)


# with pd.ExcelWriter('sgu_data_modify.xlsx') as writer:
#     sgu_data_modify.to_excel(writer, index=False)
    
sgu_data_modify = pd.read_excel('sgu_data_modify.xlsx')

# sgu_data_modify.head(2)

In [ ]:
refrence_wells_within_distance = groundwater_return_periods.filter_reference_wells_not_within_distance(observation_wells, sgu_data_modify, distance=50)

refrence_wells_within_distance.head(2)

In [ ]:
plot_groundwater_return_period(overlapping_dfs, savefig=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

def rmse(predictions, observations):
    return np.sqrt(np.mean((predictions - observations) **  2))

def area_metric_approach(predictions, observations):
    # Calculate the RMSE
    rmse_value = rmse(predictions, observations)
    
    # Calculate the observed errors as Z-scores
    observed_errors = (observations - predictions) / rmse_value
    
    # Ensure observed_errors is an array
    if not isinstance(observed_errors, np.ndarray):
        observed_errors = np.array(observed_errors)
    
    # Calculate the expected errors as Z-scores
    expected_errors = norm.ppf(0.5 +   0.5 * (1 - norm.cdf(rmse_value)))
    
    # Ensure expected_errors is an array with the same length as observed_errors
    expected_errors = np.full(observed_errors.shape, expected_errors)
    
    # Plot the observed fraction of errors vs. the expected fraction of errors
    plt.scatter(observed_errors, expected_errors, alpha=0.5)
    plt.xlabel('Observed Errors')
    plt.ylabel('Expected Errors')
    plt.title('Calibration Curve')
    plt.show()
    
    # Calculate the miscalibration area
    # This is a simplified example and does not account for all the nuances of
    # calculating the miscalibration area in a real-world scenario.
    miscalibration_area = np.abs(observed_errors - expected_errors).mean()
    
    return miscalibration_area

# Example usage:
# predictions and observations are numpy arrays or lists of equal length
predictions = np.array([1,   2,   3,   4,   5])
observations = np.array([1.1,   1.9,   3.2,   3.8,   5.1])

miscalibration_area = area_metric_approach(predictions, observations)
print(f"Miscalibration Area: {miscalibration_area}")

In [ ]:
def plot_return_periods(overlapping_dfs, savefig=False):
    
    for i in range(len(overlapping_dfs)):
        overlapping_dfs_ = pd.concat(overlapping_dfs[i])

        obs = overlapping_dfs_.loc[overlapping_dfs_.id == overlapping_dfs_.obs_id.iloc[i]]
        ref = overlapping_dfs_.loc[overlapping_dfs_.id == overlapping_dfs_.ref_id.iloc[i]]


        fig, axes = plt.subplots(nrows=1, figsize=(16, 6))
        ax1 = axes
        ax2 = ax1.twinx()

        sns.lineplot(data=obs, x='datetime', y='m asl', marker='o', ax=ax1, label=overlapping_dfs_.obs_id.iloc[i], color='blue')
        sns.lineplot(data=ref, x='datetime', y='value', marker='.', ax=ax2, label=f'{overlapping_dfs_.ref_id.iloc[i]}\nSoil type: {ref.soil_type.iloc[0]}\nAquifer: {ref.aquifer_type.iloc[0]}\nTopographic: {ref.topographic_location.iloc[0]}', color='orange')

        ax1.set_title(f'Time Series Plot for {overlapping_dfs_.obs_id.iloc[i]} and {overlapping_dfs_.ref_id.iloc[i]}')
        ax1.set_xlabel('Datetime')
        ax1.set_ylabel('m asl', color='blue', )
        ax2.set_ylabel('m asl', color='orange')

        ax1.tick_params(axis='y', labelcolor='blue')
        ax2.tick_params(axis='y', labelcolor='orange')

        corr = overlapping_dfs_.coorelation.iloc[0]
        ax1.axhline(y=obs.return_time.iloc[0], color='red', linestyle='--', label=f"Return Period 2400 year (m asl): {obs.return_time.iloc[0]:.2f}\nCorrelation (r): {corr:.2f}\nP-value: {obs.probability.iloc[0]:.2f}")


        lines, labels = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines + lines2, labels + labels2, loc='upper left', fontsize=12, bbox_to_anchor=(1.05, 0.65))

        plt.tight_layout()
        if savefig:
            plt.savefig(f"Time_Series_{overlapping_dfs_.obs_id.iloc[i]}_{overlapping_dfs_.ref_id.iloc[i]}_2400_year.jpg")
        plt.show()
        
plot_return_periods(overlapping_dfs, savefig=True)

In [ ]:
from scipy.stats import norm

df = overlapping_dfs[0][0]

def correlation_confidence_interval(corr, n):
    # Calculate standard error
    se = (1 - corr**2) / (n - 2)
    se = se**0.5

    # Set confidence level (e.g., 95%)
    confidence_level = 0.95
    alpha = 1 - confidence_level

    # Calculate critical value from the t-distribution
    cv = norm.ppf(1 - alpha / 2)

    # Calculate margin of error
    me = cv * se

    # Calculate confidence interval
    lower_bound = corr - me
    upper_bound = corr + me

    return lower_bound, upper_bound

# Example usage
sample_corr = df.coorelation.iloc[0]
sample_size = len(df)
lower, upper = correlation_confidence_interval(sample_corr, sample_size)
print(f"95% Confidence Interval: [{lower:.f}, {upper}]")




In [ ]:

def apply_filter(observation_wells, sgu_data_modify):

    distance_to_refrence_well = telecontrolnet.distance_calculation(observation_wells, sgu_data_modify)

    result = distance_to_refrence_well.drop(columns=['id', 'comment', 'm asl', 'datetime','latitude','longitude', 'quality', 'top_of_casing (m asl)', 'soil_type', 'aquifer_type', 'topographic_location'])
    columns_to_check = result.columns

    result[columns_to_check] = result[columns_to_check].map(lambda x: x if x <= 50 else pd.NA)
    result = result.dropna(axis=1, how='all')
    sgu_refrence_wells = [item.replace('distance_', '') for item in result.columns]

    refrence_wells = sgu_data_modify.loc[sgu_data_modify.id.isin(sgu_refrence_wells)].dropna(subset='value').reset_index(drop=True)


    def filter_refrence_wells_below_20_years(refrence_wells):
        refrence_wells['datetime'] = pd.to_datetime(refrence_wells['datetime'])
        grouped_data = refrence_wells.groupby('id')
        # Calculate the difference between min and max datetime for each group
        time_difference = grouped_data['datetime'].max() - grouped_data['datetime'].min()

        # Filter the groups where the time difference is equal to or greater than 20 years
        filtered_groups = time_difference[time_difference >= pd.Timedelta('7300 days')].reset_index()
        filtered_groups_list = filtered_groups.id.unique()

        refrence_wells = refrence_wells.loc[refrence_wells.id.isin(filtered_groups_list)].copy()

        return refrence_wells


    refrence_wells_ = filter_refrence_wells_below_20_years(refrence_wells)
    return refrence_wells_


refrence_wells_test = apply_filter(observation_wells, sgu_data_modify)

refrence_wells_test

In [ ]:
class groundwater_return_periods_1:
    def __init__(self):
        # self.groundwater_data = groundwater_data
        # self.precipitation_data = precipitation_data
        self.return_periods = None  # Återkomsttider kommer att lagras här efter beräkning
        
        
        # Frekvensfaktor 𝑡𝑇 för normalfördelning vid olika återkomstintervall efter Svensson & Sällfors 1985
        self.frequencyfactors = {
                                    10: 1.2816,
                                    20: 1.6449,
                                    50: 2.0538,
                                    100: 2.3264,
                                    200: 2.5758,
                                    500: 2.8782
                                }

    def create_overlapping_dataframes(self, obs_data, ref_data, correlation_threshold=0.5):
        # Convert 'datetime' to datetime type
        obs_data['datetime'] = pd.to_datetime(obs_data['datetime'])
        ref_data['datetime'] = pd.to_datetime(ref_data['datetime'])

        # Get unique IDs from observation and reference data
        unique_obs_ids = obs_data['id'].unique()
        np.random.shuffle(unique_obs_ids)
        unique_ref_ids = ref_data['id'].unique()

        # Create a list to store overlapping dataframes
        overlapping_dfs = []

        # Check for overlapping date range for each unique ID
        for obs_id in unique_obs_ids:
            print(f"Observation well: {obs_id}")
            
            obs_date_range = obs_data[obs_data['id'] == obs_id]['datetime']

            # Drop any missing values before creating the IntervalIndex
            obs_date_range = obs_date_range.dropna()

            obs_start = obs_date_range.min()
            obs_end = obs_date_range.max()

            for ref_id in unique_ref_ids:
                
                ref_date_range = ref_data[ref_data['id'] == ref_id]['datetime']

                # Drop any missing values before creating the IntervalIndex
                ref_date_range = ref_date_range.dropna()

                ref_start = ref_date_range.min()
                ref_end = ref_date_range.max()

                overlap = not (obs_end < ref_start or obs_start > ref_end)

                if overlap:
                    obs_df = obs_data[obs_data['id'] == obs_id]
                    ref_df = ref_data[ref_data['id'] == ref_id]

                    aligned_df = pd.merge_asof(
                        obs_df.sort_values('datetime'),
                        ref_df.sort_values('datetime'),
                        on='datetime'
                    )
                    pd.set_option('future.no_silent_downcasting', True)
                    aligned_df = aligned_df.replace([np.inf, -np.inf], np.nan).dropna()
#                     aligned_df = aligned_df.infer_objects(copy=False)
                    corr = None
                    
                    # Check if both arrays have at least two data points
                    if len(aligned_df['value']) >= 2 and len(aligned_df['m asl']) >= 2:
                        corr, _ = pearsonr(aligned_df['value'], aligned_df['m asl'])
                        
#                         print(f"Observation well: {obs_id} and refrence well: {ref_id}\nCorr: {corr}")
                    

                    if corr is not None and corr > correlation_threshold:
                        
                        obs_df = obs_df.copy()
                        ref_df = ref_df.copy()
                        obs_df['coorelation'] = corr
                        ref_df['coorelation'] = corr
                        filtered_obs, filtered_ref = self.filter_data(obs_df, ref_df)
                    
                        if filtered_obs is not None and filtered_ref is not None:
                            
                            return_time, standard_deviation, max_values_by_year, frequencyfactors, Y_RmaxT, S_tr = self.process_observed_values(filtered_ref, filtered_obs)
                            filtered_obs.loc[:, 'ref_id'] = ref_id
                            filtered_obs.loc[:, 'obs_id'] = obs_id
                            filtered_obs.loc[:, 'return_time'] = return_time                     
                            filtered_obs.loc[:, 'standard_deviation'] = standard_deviation
                            
                            filtered_obs.loc[:, 'Y_RmaxT'] = Y_RmaxT
                            filtered_obs.loc[:, 'S_tr'] = S_tr
                            
#                             print(f"filtered_obs {filtered_obs}")
#                             print(f"filtered_ref {filtered_ref}")
                            
                            overlapping_dfs.append((filtered_obs, filtered_ref))
#                             overlapping_dfs.append(filtered_obs)
        
#         overlapping_dfs = pd.concat(overlapping_dfs)
        
        return overlapping_dfs

    def process_observed_values(self, referens_data, observation_data, hydrological_start_month=10, return_time=50):

        """
        Processes observed values of the reference well.

        Parameters:
        - referens_data: DataFrame with columns 'datetime', 'value'.
        - observation_data: DataFrame with columns 'datetime', 'value'.
        - hydrological_start_month: Start month of the hydrological year (October).

        Returns:
        - plotting_positions: DataFrame with columns 'rank', 'max_value', 'probability'.
        """

        # Convert 'datetime' to datetime format
        referens_data['datetime'] = pd.to_datetime(referens_data['datetime'])
        observation_data['datetime'] = pd.to_datetime(observation_data['datetime'])

        # Extract hydrological year from the observation date
        referens_data['hydrological_year'] = np.where(referens_data['datetime'].dt.month >= hydrological_start_month,
                                                    referens_data['datetime'].dt.year,
                                                    referens_data['datetime'].dt.year - 1)


        observation_data['hydrological_year'] = np.where(observation_data['datetime'].dt.month >= hydrological_start_month,
                                                    observation_data['datetime'].dt.year,
                                                    observation_data['datetime'].dt.year - 1)
        
        # Group data by hydrological year and find the maximum value in each year
        max_values_by_year = referens_data.groupby('hydrological_year')['value'].max().reset_index()


        # Sort values in descending order to rank them
        max_values_by_year = max_values_by_year.sort_values(by='value', ascending=False).reset_index(drop=True)

        # Assign ranks to sorted values
        max_values_by_year['rank'] = max_values_by_year.index + 1

        # Calculate plotting positions using Weibull formula
        max_values_by_year['probability'] = (len(max_values_by_year) + 1 - max_values_by_year['rank']) / (len(max_values_by_year) + 1)
        
        # Calculate return period using the provided formula (equation 7)
        max_values_by_year['return_period'] = 1 / max_values_by_year['probability']

        # Create a DataFrame with the required columns
        plotting_positions = max_values_by_year[['rank', 'value', 'probability']].rename(columns={'value': 'max_value'})

        standard_deviation = Smax = plotting_positions['max_value'].std()
        
        frequencyfactors = self.frequencyfactors.get(return_time)
        
#         print(f"frequencyfactors {frequencyfactors}")
        
        Y_RmaxT = plotting_positions['max_value'].mean() + frequencyfactors * Smax

        S_tr = Y_RmaxT - plotting_positions['max_value'].max()

        return_time = Y0_max_T = observation_data['m asl'].max() + S_tr * ((observation_data['m asl'].max() - observation_data['m asl'].min()) / (plotting_positions['max_value'].max() - plotting_positions['max_value'].min()))


        referens_data_id = referens_data.id.unique()[0]
        observation_data_id = observation_data.id.unique()[0]
        
#         print('referens well: ', referens_data_id
#               , '\nobservation well: ', observation_data_id
#               , '\nReturn time 50 years: ', Y0_max_T
#               )
        
        return return_time, standard_deviation, max_values_by_year, frequencyfactors, Y_RmaxT, S_tr
    
    def filter_data(self, observations_data, reference_data):
        """
        Filters data based on specified criteria for reference and observation datasets.

        Parameters:
        - referens_data: DataFrame with columns 'observation_date', 'variations', and 'reference_lifetime'.
        - observations_data: DataFrame with columns 'observation_date', 'variations'.

        Returns:
        - filtered_referens: DataFrame after applying filters and interpolation for reference data.
        - filtered_observations: DataFrame after applying filters and interpolation for observation data.
        """
        

    # Convert 'datetime' to datetime format for both datasets
        reference_data['datetime'] = pd.to_datetime(reference_data['datetime'])
        observations_data['datetime'] = pd.to_datetime(observations_data['datetime'])

        # Set the date frequency to 'MS' (Month Start) to check for monthly observations for both datasets
        reference_data.set_index('datetime', inplace=True)
        observations_data.set_index('datetime', inplace=True)

        # Identify numeric columns for resampling
        numeric_columns_ref = reference_data.select_dtypes(include=[np.number]).columns
        numeric_columns_obs = observations_data.select_dtypes(include=[np.number]).columns


        # Resample only numeric columns
    #     print(reference_data)
        reference_data_resampled = reference_data[numeric_columns_ref].resample('MS').max()
        observations_data_resampled = observations_data[numeric_columns_obs].resample('MS').max()

        # Calculate the lifetime variation range for reference and observation datasets
        lifetime_variation_range_reference = reference_data_resampled['value'].max() - reference_data_resampled['value'].min()
        lifetime_variation_range_observation = observations_data_resampled['m asl'].max() - observations_data_resampled['m asl'].min()

        
        # Set the maximum allowed variation as 30% of the variation range for the entire lifetime
        max_variation_observation = 0.3 * lifetime_variation_range_reference
        print(f'Ref {reference_data.id.iloc[0]}: {lifetime_variation_range_reference}\nObs {observations_data.id.iloc[0]}: {lifetime_variation_range_observation}\nMax variation observation: {max_variation_observation}')

        # Check if the observation well's variation is 30% higher or more than the reference well
#         if lifetime_variation_range_observation >= 1.3 * lifetime_variation_range_reference:
        if lifetime_variation_range_observation <= max_variation_observation:
            print(f"Reference well {reference_data.id.iloc[0]} has too high variation compared to the observation well {observations_data.id.iloc[0]}.")
            return None, None
        else:
            # Filter data based on variations for reference_data
            filtered_reference = reference_data[abs(reference_data['value'] - reference_data['value'].mean()) <= max_variation_observation].copy()
            
            filtered_reference = filtered_reference.infer_objects(copy=False).interpolate(method='linear')
            
#             observations_data = observations_data.infer_objects(copy=False).interpolate(method='linear')
            filtered_observations = observations_data.infer_objects(copy=False).interpolate(method='linear')
            
            # Linear interpolation for missing data for reference_data
#             filtered_reference['value'] = filtered_reference['value'].astype(float)
#             filtered_observations['m asl'] = filtered_observations['m asl'].astype(float)

            
            # Reset index for final result for both datasets
            filtered_reference.reset_index(inplace=True)
            filtered_observations.reset_index(inplace=True)

            return filtered_observations, filtered_reference
    
groundwater_return_periods = groundwater_return_periods_1()
# observation_wells = observation_wells.reset_index()
# refrence_wells = refrence_wells.reset_index()


overlapping_dfs = groundwater_return_periods.create_overlapping_dataframes(observation_wells, refrence_wells)

In [ ]:
for i in range(len(overlapping_dfs)):
    overlapping_dfs_ = pd.concat(overlapping_dfs[i])
    
    obs = overlapping_dfs_.loc[overlapping_dfs_.id == overlapping_dfs_.obs_id.iloc[i]]
    ref = overlapping_dfs_.loc[overlapping_dfs_.id == overlapping_dfs_.ref_id.iloc[i]]
                                                                             
    
    fig, axes = plt.subplots(nrows=1, figsize=(16, 6))
    ax1 = axes
    ax2 = ax1.twinx()

    sns.lineplot(data=obs, x='datetime', y='m asl', marker='o', ax=ax1, label=overlapping_dfs_.obs_id.iloc[i], color='blue')
    sns.lineplot(data=ref, x='datetime', y='value', marker='.', ax=ax2, label=overlapping_dfs_.ref_id.iloc[i], color='orange')

    ax1.set_title(f'Time Series Plot for {overlapping_dfs_.obs_id.iloc[i]} and {overlapping_dfs_.ref_id.iloc[i]}')
    ax1.set_xlabel('Datetime')
    ax1.set_ylabel('m asl', color='blue', )
    ax2.set_ylabel('m asl', color='orange')

    ax1.tick_params(axis='y', labelcolor='blue')
    ax2.tick_params(axis='y', labelcolor='orange')
    
    corr = overlapping_dfs_.coorelation.iloc[0]
    ax1.axhline(y=obs.return_time.iloc[0], color='red', linestyle='--', label=f"Return time 50 year: {obs.return_time.iloc[0]:.2f}\nCorrelation (r): {corr:.2f}")


    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper left', fontsize=12, bbox_to_anchor=(1.05, 0.65))
    
    plt.tight_layout()
    plt.savefig(f"Time_Series_{overlapping_dfs_.obs_id.iloc[i]}_{overlapping_dfs_.ref_id.iloc[i]}.jpg")
    plt.show()
    
    
    


In [ ]:

overlapping_dfs = groundwater_return_periods.create_overlapping_dataframes(observation_wells, refrence_wells)

overlapping_dfs

In [ ]:
def plot_sgu_and_telecontrolnet_meter(df_sgu, df_telecontrolnet, coorelation_value=0.7):
            warnings.filterwarnings("ignore", "is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead.")
            warnings.filterwarnings("ignore", "use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.")

            sns.set_theme()

            # Plotting on individual figures
            unique_ids = df_telecontrolnet['id'].unique()
            unique_sgu_ids = df_sgu['id'].unique()

            for nr in unique_sgu_ids:
                subset = df_sgu.loc[df_sgu.id == nr]
                
                subset.loc[:,'datetime'] = pd.to_datetime(subset['datetime'], format='%Y-%m-%d %H:%M:%S')
                subset.set_index('datetime', inplace=True)

                # Resample 'm asl' to daily values using mean aggregation
                resampled_sgu = subset['value'].resample('D').mean().reset_index()
                resampled_sgu['datetime'] = pd.to_datetime(resampled_sgu['datetime'].dt.strftime('%Y-%m-%d') + ' 12:00:00')

                # Convert the 'datetime' column back to datetime type
                resampled_sgu['datetime'] = pd.to_datetime(resampled_sgu['datetime'], format='%Y-%m-%d %H:%M:%S')
                resampled_sgu['id'] = nr
                resampled_sgu = resampled_sgu.dropna()
                
            
                for i, unique_id in enumerate(unique_ids):
                    
                    subset_df = df_telecontrolnet[df_telecontrolnet['id'] == unique_id]
                    subset_df.loc[:,'datetime'] = pd.to_datetime(subset_df['datetime'], format='%Y-%m-%d %H:%M:%S')
                    subset_df.set_index('datetime', inplace=True)

                    # Resample 'm asl' to daily values using mean aggregation
                    resampled_df = subset_df['value'].resample('D').mean().reset_index()
                    resampled_df['datetime'] = pd.to_datetime(resampled_df['datetime'].dt.strftime('%Y-%m-%d') + ' 12:00:00')

                    # Convert the 'datetime' column back to datetime type
                    resampled_df['datetime'] = pd.to_datetime(resampled_df['datetime'], format='%Y-%m-%d %H:%M:%S')
                    resampled_df['id'] = unique_id


                    resampled_sgu.loc[:,'datetime'] = pd.to_datetime(resampled_sgu['datetime'], format='%Y-%m-%d %H:%M:%S')
                                   
                    
                    aligned_df = pd.merge_asof(
                        resampled_sgu.sort_values('datetime'),
                        resampled_df.sort_values('datetime'),
                        on='datetime'
                    )
                    
                    print(aligned_df)
                    # Check if 'aligned_df' DataFrame is not empty before calculating correlation
                    if len(aligned_df) > 0 and len(aligned_df['value'].dropna()) > 1 and len(aligned_df['value'].dropna()) > 1:
                        # Calculate correlation and standardize it
                        aligned_df = aligned_df.replace([np.inf, -np.inf], np.nan).dropna()
                        corr, _ = pearsonr(aligned_df['value'], aligned_df['value'])
                        
                
                        if corr > coorelation_value:
                            fig, axes = plt.subplots(nrows=1, figsize=(16, 6))
                            ax1 = axes
                            ax2 = ax1.twinx()

                            sns.lineplot(data=resampled_df, x='datetime', y='value', marker='o', ax=ax1, label=unique_id, color='blue')
                            sns.lineplot(data=resampled_sgu, x='datetime', y='value', marker='.', ax=ax2, label=nr, color='orange')

                            ax1.set_title(f'Time Series Plot for {unique_id}')
                            ax1.set_xlabel('Datetime')
                            ax1.set_ylabel('m asl', color='blue')
                            ax2.set_ylabel('m asl', color='orange')

                            ax1.tick_params(axis='y', labelcolor='blue')
                            ax2.tick_params(axis='y', labelcolor='orange')

                            lines, labels = ax1.get_legend_handles_labels()
                            lines2, labels2 = ax2.get_legend_handles_labels()
                            ax2.legend(lines + lines2, labels + labels2, loc='upper right', fontsize=12)
                            
                            
                            textstr = f'Correlation (r): {corr:.2f}\nR-squared: {corr**2:.2f}'
                            ax2.text(0.05, 0.95, textstr, transform=ax1.transAxes, fontsize=12,
                                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))



                plt.tight_layout()
                plt.show()
                
plot_sgu_and_telecontrolnet_meter(obs_df, refrence_wells)
            


In [ ]:
# access_token, tags_mwr, tags_temp = telecontrolnet.accesstoken()
# telecontrolnet_level_data= telecontrolnet.get_data(access_token, tags_mwr)
# telecontrolnet_temprature_data = telecontrolnet.get_data(access_token, tags_temp)
# telecontrolnet_modify = telecontrolnet.megre_frames(tags_mwr,  telecontrolnet_temprature_data, telecontrolnet_level_data)

# telecontrolnet_distance_to_refrence_well = telecontrolnet.distance_calculation(telecontrolnet_modify, sgu_data_modify)

# # telecontrolnet_metadata = telecontrolnet_distance[['station_id','id', 'datetime','m asl','°C','latitude','longitude', 'city','active']].copy()
# telecontrolnet_result = telecontrolnet_distance_to_refrence_well.drop(columns=['station_id','id', 'datetime','m asl','°C','latitude','longitude', 'city','active'])

# columns_to_check = telecontrolnet_result.columns

# telecontrolnet_distance_to_refrence_well[columns_to_check] = telecontrolnet_distance_to_refrence_well[columns_to_check].map(lambda x: x if x <= 50 else pd.NA)
# telecontrolnet_distance_to_refrence_well = telecontrolnet_distance_to_refrence_well.dropna(axis=1, how='all')


# refrence_wells = telecontrolnet_distance_to_refrence_well.drop(columns=['station_id','id', 'datetime','m asl','°C','latitude','longitude', 'city','active']).columns.to_list()
# sgu_refrence_wells = [item.replace('distance_', '') for item in refrence_wells]
